In [ ]:
!pip install ifcopenshell


In [ ]:
from google.colab import files

print("📁 Please upload your IFC file...")
uploaded = files.upload()  # Opens file picker in Colab

# Get the uploaded file name
ifc_path = list(uploaded.keys())[0]
print(f"✅ Loaded file: {ifc_path}")

📁 Please upload your IFC file...


In [ ]:
import ifcopenshell
# Load the IFC file
model = ifcopenshell.open(ifc_path)



In [ ]:
import ifcopenshell.geom
import ifcopenshell.util.shape
from typing import Any, Callable, Optional, Union, Literal, overload
from collections.abc import Generator, Sequence
from collections import namedtuple
import numpy as np
from collections import defaultdict
import pandas as pd

print(ifcopenshell.version)

0.8.4.post1


In [ ]:
# List entity types
types = set(ent.is_a() for ent in model)
print("Entity types found:\n" + "\n".join(sorted(types)))


# Geometry-based area/volume calculation
def compute_area_volume(entity, model):
    try:
        settings = ifcopenshell.geom.settings()
        settings.set(settings.USE_WORLD_COORDS, True)
        shape = ifcopenshell.geom.create_shape(settings, entity)
    except RuntimeError:
        return 0.0, 0.0  # no geometry available

    import numpy as np
    verts = np.array(shape.geometry.verts).reshape(-1, 3)
    faces = np.array(shape.geometry.faces).reshape(-1, 3)

    area = 0.0
    volume = 0.0
    for f in faces:
        v1, v2, v3 = verts[f[0]], verts[f[1]], verts[f[2]]
        area += np.linalg.norm(np.cross(v2 - v1, v3 - v1)) / 2
        volume += np.dot(v1, np.cross(v2, v3)) / 6

    return abs(area), abs(volume)

# Unified report for walls, floors, and roofs
def sum_area_volume_entities(model):
    totals = defaultdict(lambda: {"area": 0.0, "volume": 0.0})

    # Walls
    for wall in model.by_type("IfcWall") + model.by_type("IfcWallStandardCase"):
        obj_type = getattr(wall, "ObjectType", None) or "Undefined"
        area, volume = compute_area_volume(wall, model)
        totals[f"Wall - {obj_type}"]["area"] += area
        totals[f"Wall - {obj_type}"]["volume"] += volume

    # Floors (IfcSlab)
    for slab in model.by_type("IfcSlab"):
        obj_type = getattr(slab, "ObjectType", None) or "Undefined"
        area, volume = compute_area_volume(slab, model)
        totals[f"Floor - {obj_type}"]["area"] += area
        totals[f"Floor - {obj_type}"]["volume"] += volume

    # Roofs (IfcRoof)
    for roof in model.by_type("IfcRoof"):
        obj_type = getattr(roof, "ObjectType", None) or "Undefined"
        area, volume = compute_area_volume(roof, model)
        totals[f"Roof - {obj_type}"]["area"] += area
        totals[f"Roof - {obj_type}"]["volume"] += volume

    # Columns (IfcColumn)
    for column in model.by_type("IfcColumn"):
        obj_type = getattr(column, "ObjectType", None) or "Undefined"
        area, volume = compute_area_volume(column, model)
        totals[f"Column - {obj_type}"]["area"] += area
        totals[f"Column - {obj_type}"]["volume"] += volume

    # Beams (IfcBeams)
    for beam in model.by_type("IfcBeam"):
        obj_type = getattr(beam, "ObjectType", None) or "Undefined"
        area, volume = compute_area_volume(beam, model)
        totals[f"Beam - {obj_type}"]["area"] += area
        totals[f"Beam - {obj_type}"]["volume"] += volume

    # Print results
    for obj_type, q in totals.items():
        print(f"{obj_type}")
        print(f"  Total Area, m2: {q['area']:.2f}")
        print(f"  Total Volume, m3: {q['volume']:.2f}")
        print("-" * 40)

    # Count windows
    windows = defaultdict(int)
    for window in model.by_type("IfcWindow"):
        obj_type = getattr(window, "ObjectType", None) or "Undefined"
        windows[obj_type] += 1

    print("Window counts by ObjectType:")
    for obj_type, count in windows.items():
        print(f"  {obj_type}: {count}")

    # Count doors
    doors = defaultdict(int)
    for door in model.by_type("IfcDoor"):
        obj_type = getattr(door, "ObjectType", None) or "Undefined"
        doors[obj_type] += 1

    print("Door counts by ObjectType:")
    for obj_type, count in doors.items():
        print(f"  {obj_type}: {count}")

# Convert to DataFrame
    df = pd.DataFrame([
        {"Element/ObjectType": name, "Total Area, m2": q["area"], "Total Volume, m3": q["volume"]}
        for name, q in totals.items()
    ])

    # Add window and door counts as extra rows
    windows = len(model.by_type("IfcWindow"))
    doors = len(model.by_type("IfcDoor"))

    df = pd.concat([
        df,
        pd.DataFrame([
            {"Element/ObjectType": "Windows", "Count": windows},
            {"Element/ObjectType": "Doors", "Count": doors}
        ])
    ], ignore_index=True)

    # Save to Excel
    df.to_excel("aggregated_quantities.xlsx", index=False)

    return df

# Run it
report_df = sum_area_volume_entities(model)

# Download in Colab
files.download("aggregated_quantities.xlsx")

report_df

"""
# Run it
sum_area_volume_entities(model)


# Count windows
windows = model.by_type("IfcWindow")
print("Number of windows:", len(windows))

# Count doors
doors = model.by_type("IfcDoor")
print("Number of doors:", len(doors))



#Use this code if you want to check qauntities by one object
#Code is adaptable for each IfcEntity:
for column in model.by_type("IfcColumn"):
    area, volume = compute_area_volume(column, model)
    print(f"Entity: {column.is_a()} | Name: {column.ObjectType}")
    print(f"  Area: {area:.2f}")
    print(f"  Volume: {volume:.2f}")
    print("-" * 40)
"""

